In [3]:
from typing import TypedDict
import requests
import os
from langgraph.graph import StateGraph, START, END
SAIS_APP_URL = os.getenv("SAIS_APP_URL")
SAIS_TOKEN = os.getenv("SAIS_TOKEN")
SAIS_MODEL = os.getenv("SAIS_MODEL", "gpt-4.1")

if not SAIS_APP_URL or not SAIS_TOKEN:
    raise SystemExit("Error: SAIS_APP_URL and SAIS_TOKEN environment variables must be set.")

PROXY_HOST = os.getenv("PROXY_HOST")
PROXY_PORT = os.getenv("PROXY_PORT")
PROXY_ENABLED = os.getenv("PROXY_ENABLED", "false").lower() == "true"
SSL_VERIFY = os.getenv("SSL_VERIFY", "true").lower() == "true"


In [5]:
def generate_answer(context: str, query: str) -> str:
    response = requests.post(
        f"{SAIS_APP_URL}/v1/responses",
        headers={
            "Authorization": f"Bearer {SAIS_TOKEN}",
            "Content-Type": "application/json",
            "ApplicationType": "BRProduct",
        },
        json={
            "model": SAIS_MODEL,
            "instructions": "You are an AI technical support assistant.\n\nAnswer the user's question to the best of your knowledge.",
            "input": context + "\n\n" + query,
        },
        verify=SSL_VERIFY,
        proxies={
            "http": f"http://{PROXY_HOST}:{PROXY_PORT}" if PROXY_ENABLED else None,
            "https": f"http://{PROXY_HOST}:{PROXY_PORT}" if PROXY_ENABLED else None,
        }
    )
    if response.status_code == 200:
        response_data = response.json()
        try:
            outputs = response_data["body"]["output"]
            texts = []
            for item in outputs:
                if item.get("type") == "message":
                    for content in item.get("content", []):
                        if content.get("type") == "output_text":
                            texts.append(content.get("text", ""))
            if texts:
                return "\n".join(texts)
            return "Error: No output text found in the response."
        except KeyError:
            return "Error: Unexpected response format."
    else:
        return f"Error: Request failed with status code {response.status_code}"

In [7]:
class LLMcallState(TypedDict):
    prompt: str
    response: str

def call_llm(state: LLMcallState) -> LLMcallState:
    prompt = state["prompt"]
    response = generate_answer("", prompt)
    state["response"] = response
    return state

graph = StateGraph(LLMcallState)
graph.add_node("call_llm", call_llm)
graph.set_entry_point("call_llm")
graph.add_edge("call_llm", END)

app= graph.compile()

query = input("Enter your question: ")

initial_state = {
    "prompt": query,
    "response": ""
}
result = app.invoke(initial_state)

print("Prompt:", result["prompt"])
print("Response:", result["response"])  



c:\Users\rajpri\Documents\GenerativeAI\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.98.21.23'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Prompt: What is mortgage backed security 
Response: A **mortgage-backed security (MBS)** is an investment made up of a pool of home loans (mortgages).

Here’s how it works:

1. A bank or lender gives mortgages to homebuyers.
2. Those mortgages are bundled together.
3. The bundle is sold to investors as a security.
4. Homeowners make monthly mortgage payments.
5. Investors receive payments from the interest and principal collected from those mortgages.

In simple terms, buying an MBS means you are investing in the cash flows from many people’s mortgage payments.

### Example
If 1,000 homeowners each pay their monthly mortgage, those payments are pooled and passed on to investors who own the mortgage-backed security.

### Types of MBS
- **Agency MBS**: Backed or guaranteed by government-related agencies such as Ginnie Mae, Fannie Mae, or Freddie Mac.
- **Non-agency MBS**: Issued by private institutions and usually carry more credit risk.

### Main risks
- **Prepayment risk**: Homeowners 

In [9]:
query = ""
while query != "exit":
    query = input("Enter your question: ")
    if query == "exit":
        break
    initial_state = {
        "prompt": query,
        "response": ""
    }
    result = app.invoke(initial_state)
    print("Prompt:", result["prompt"])
    print("Response:", result["response"])

c:\Users\rajpri\Documents\GenerativeAI\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.98.21.23'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Prompt: How can i buld and call tools in langgraph ?
Response: Here’s the basic pattern for **building and calling tools in LangGraph**.

## 1. Install packages

```bash
pip install langgraph langchain langchain-openai
```

You also need your model API key, for example:

```bash
export OPENAI_API_KEY="your-key"
```

---

## 2. Define tools

LangGraph uses LangChain tools. The easiest way is with the `@tool` decorator.

```python
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together."""
    return a * b

tools = [add, multiply]
```

Important: always include a docstring. The LLM uses it to understand when to call the tool.

---

## 3. Bind tools to your LLM

```python
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

llm_with_tools = llm.bind_tools(tools)
```

`bind_tools()` lets the model decide when t